# Real Estate Prompt Agent — Colab Notebook

This notebook contains the full updated Agno pipeline for a real-estate social media prompt agent with:

- structured analyst → crafter → verifier flow
- modular skills
- project/brand/tool retrieval
- conflict handling
- prompt verification before final output

## Before you run it
Set `OPENAI_API_KEY` either:
- as an environment variable, or
- as a Colab secret named `OPENAI_API_KEY`


## 1. Install (run in Colab/Jupyter only if needed)

In [158]:
!pip install -U agno openai pydantic

In [159]:
from __future__ import annotations

import json
import os
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Optional

from pydantic import BaseModel, Field

from agno.agent import Agent
from agno.models.openrouter import OpenRouterResponses
from agno.skills import LocalSkills, Skills
# Removed: from agno.tools import tool  (not needed — plain functions work as tools)

In [160]:
os.environ["OPENAI_API_KEY"] = "replace-with-your-openai-api-key"  # Replace with your key
os.environ["OPENROUTER_API_KEY"] = "replace-with-your-openrouter-api-key"  # Replace with your key


## 2. API key setup (safe)

In [161]:
def ensure_openai_api_key() -> None:
    """Load OPENAI_API_KEY from environment or Colab secrets."""
    if os.getenv("OPENAI_API_KEY"):
        return

    # Colab secret path
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            return
    except Exception:
        pass

    raise RuntimeError(
        "OPENAI_API_KEY not found. Set it in environment variables or as a Colab secret named OPENAI_API_KEY."
    )


ensure_openai_api_key()

def ensure_openrouter_api_key() -> None:
    """Load OPENROUTER_API_KEY from environment or Colab secrets."""
    if os.getenv("OPENROUTER_API_KEY"):
        return

    # Colab secret path
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("OPENROUTER_API_KEY")
        if key:
            os.environ["OPENROUTER_API_KEY"] = key
            return
    except Exception:
        pass

    raise RuntimeError(
        "OPENROUTER_API_KEY not found. Set it in environment variables or as a Colab secret named OPENROUTER_API_KEY."
    )


ensure_openrouter_api_key()

## 3. File structure

In [162]:
BASE_DIR = Path(".")
SKILLS_DIR = BASE_DIR / "skills"
DATA_DIR = BASE_DIR / "data"

for p in [SKILLS_DIR, DATA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

In [163]:
"""## 3A. Optional visual uploads"""

import json
from pathlib import Path

INPUTS_DIR = BASE_DIR / "inputs"
INPUTS_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_UPLOAD_SLOTS = {
    "reference_image": "reference_image",
    "template_image": "template_image",
    "logo_image": "logo_image",
}

def _slot_matches(slot_name: str) -> List[Path]:
    return sorted(INPUTS_DIR.glob(f"{slot_name}.*"))

def _existing_slot_path(slot_name: str) -> Optional[str]:
    matches = _slot_matches(slot_name)
    return str(matches[0]) if matches else None

def _clear_slot(slot_name: str) -> None:
    for p in _slot_matches(slot_name):
        p.unlink(missing_ok=True)

def _save_uploaded_file_to_slot(slot_name: str, original_name: str, content: bytes) -> str:
    ext = Path(original_name).suffix.lower().strip()
    if not ext:
        ext = ".png"

    canonical_stem = CANONICAL_UPLOAD_SLOTS[slot_name]
    dest = INPUTS_DIR / f"{canonical_stem}{ext}"

    _clear_slot(slot_name)
    dest.write_bytes(content)
    return str(dest)

def upload_visual_slot(slot_name: str, enabled: bool = False) -> Optional[str]:
    """
    Upload exactly one file for a slot and save it with a canonical filename:
    - reference_image.<ext>
    - template_image.<ext>
    - logo_image.<ext>

    If enabled=False, returns any existing canonical file for that slot.
    """
    if not enabled:
        return _existing_slot_path(slot_name)

    try:
        from google.colab import files  # type: ignore
    except Exception as e:
        raise RuntimeError("This upload helper is intended for Google Colab.") from e

    print(f"Upload one file for: {slot_name}")
    uploaded = files.upload()

    if not uploaded:
        return _existing_slot_path(slot_name)

    if len(uploaded) != 1:
        raise ValueError(f"Please upload exactly one file for {slot_name}.")

    original_name, content = next(iter(uploaded.items()))
    saved_path = _save_uploaded_file_to_slot(slot_name, original_name, content)
    print(f"Saved to: {saved_path}")
    return saved_path


# Toggle these when you want to upload / replace a slot.
UPLOAD_REFERENCE_IMAGE = False
UPLOAD_TEMPLATE_IMAGE = False
UPLOAD_LOGO_IMAGE = True

REFERENCE_IMAGE_PATH = upload_visual_slot("reference_image", enabled=UPLOAD_REFERENCE_IMAGE)
TEMPLATE_IMAGE_PATH = upload_visual_slot("template_image", enabled=UPLOAD_TEMPLATE_IMAGE)
LOGO_IMAGE_PATH = upload_visual_slot("logo_image", enabled=UPLOAD_LOGO_IMAGE)

# Optional notes used by the example run below.
REFERENCE_IMAGE_NOTE = (
    "Use vision to understand the reference image, choose the strongest hero, "
    "and decide crop / angle / realism treatment while preserving identity."
)

TEMPLATE_ID = None  # e.g. 'documentary_overlay_update' or 'organic_shape_launch'
TEMPLATE_NOTE = (
    "Use the template image only as a style / composition cue. "
    "Do not let it override the project identity."
)

LOGO_NOTE = (
    "Reproduce the uploaded logo exactly and place it with disciplined hierarchy."
)

print(json.dumps({
    "REFERENCE_IMAGE_PATH": REFERENCE_IMAGE_PATH,
    "TEMPLATE_IMAGE_PATH": TEMPLATE_IMAGE_PATH,
    "LOGO_IMAGE_PATH": LOGO_IMAGE_PATH,
    "TEMPLATE_ID": TEMPLATE_ID,
}, indent=2))

Upload one file for: logo_image


Saving icon.png to icon (1).png
Saved to: inputs/logo_image.png
{
  "REFERENCE_IMAGE_PATH": null,
  "TEMPLATE_IMAGE_PATH": null,
  "LOGO_IMAGE_PATH": "inputs/logo_image.png",
  "TEMPLATE_ID": null
}


## 4. Skills

In [164]:
"""## 4. Skills (style-first consolidated)"""

def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.strip() + "\n", encoding="utf-8")


# ============================================================
# SKILL 1: briefly-social-core
# ============================================================
briefly_social_core_md = r"""
---
name: briefly-social-core
description: Core operating system for Briefly Social. Separates post intent from design style, defines priority order, handles external visual inputs, and sets finished-poster defaults.
metadata:
  version: "5.1.0"
  tags: ["real-estate", "social-media", "core", "style-first", "poster-system"]
---

# Briefly Social Core

Use this skill for every Briefly Social creative task.

## Mission
Create a brand-right, asset-grounded, poster-grade real-estate social creative that:
- preserves factual and visual truth
- matches the communication job of the post
- chooses a deliberate design style instead of defaulting to repeated layouts
- uses a structured visual system rather than "nice building image + text"
- feels like a finished campaign asset when finished_poster mode is expected

## Fundamental Model
post = communication_job × payload × asset_mode × style_family × style_modifiers × lever_bundle × truth_constraints

## Critical Separation
Do not confuse these:

### 1) Post Type / Communication Job
This answers: what is the post trying to do?
Examples:
- project_launch
- construction_update
- festival_post
- amenity_spotlight
- site_visit_invite
- testimonial
- ad

### 2) Design Style
This answers: how should the creative visually behave?
A post type does not own a single style.
Any suitable post type can use different style families if the payload, asset, and market tone fit.

### 3) Asset Mode
This answers: what visual anchor is available?
Examples:
- single tower
- villa
- township aerial
- amenity
- candid site presence
- symbolic graphic-led

### 4) Payload Architecture
This answers: how text/data must be organized.
Examples:
- slogan-first
- scarcity-first
- footer-heavy
- quote-led
- one-statement poster

## Priority Order
1. asset truth / project truth / compliance truth
2. explicit brief instructions
3. communication job of the post
4. post-type contract
5. chosen style family
6. chosen lever settings
7. brand truth
8. creative flourish

## External Visual Inputs

### reference image
Treat a reference image as an identity or scene anchor.
Use vision to understand:
- what the image actually shows
- whether the strongest hero is one building, many buildings, an amenity, or a candid scene
- whether a tighter crop, lower angle, clearer hierarchy, or safer realism upgrade will help

Allowed improvements:
- crop
- emphasis shift
- angle refinement
- realism / lighting polish
- atmosphere
- editorial finish

Not allowed:
- changing architecture identity
- changing construction state
- inventing towers, amenities, or scale
- using the template look to override the reference truth

### template image
Treat a template image as a style and composition cue only.
It may influence:
- layout behavior
- graphic layer language
- type mood
- density
- poster finish

It must not:
- override project identity
- replace the reference image as truth anchor
- force the wrong post behavior
- create a copy-paste imitation when the asset fit is weak

### logo image
Treat a logo image as an exact brand mark.
Preserve:
- symbol shape
- lockup
- relative balance
- recognisable character

Do not:
- stylize
- rewrite
- distort
- invent alternate marks

## Universal Rules
1. Never invent project facts.
2. Never swap project identity.
3. Never treat an unavailable asset as if it exists.
4. Never use premium styling to justify generic luxury clichés.
5. Never beautify a real construction or site image to the point of falsification.
6. Never choose a visually interesting direction that weakens the post's actual job.
7. Never compensate for weak inputs by inventing unverified specifics.
8. Never force the same composition across all projects.
9. If a logo image is provided and exact fidelity is required, reproduce it exactly and keep the lockup intact.
10. Keep the post anchored around one dominant selling idea unless the brief explicitly requires a multi-message layout.
11. Never default to a plain hero image with normal text overlay.
12. Never solve a social poster by simply placing a building image and adding a generic headline.
13. Do not use empty sky or generic clean space as a substitute for real poster composition.
14. Poster-grade outputs must show deliberate hierarchy, composition logic, and design behavior.
15. If the result could be described as "just a nice render with text", it is a failure.
16. Design style must be selected intentionally and explained through visible composition choices.
17. Do not tie style too rigidly to post type; tie it to the job, asset, and payload fit.
18. When a reference image exists, truth follows the reference image first unless the brief explicitly says otherwise.
19. When a template image exists, style follows the template only as far as truth allows.
20. When a logo image exists, logo fidelity is stricter than creative flourish.

## Delivery Strategy
Choose one:
- finished_poster
- base_visual

### Default Rule
For Briefly Social, default to:
- delivery_mode = finished_poster

Use `base_visual` only when the brief explicitly asks for:
- clean text-safe background only
- background plate
- no in-image copy
- render to be designed later elsewhere
- base hero asset only

### Finished Poster Rules
If delivery_mode = finished_poster:
- style_family is mandatory
- lever bundle is mandatory
- copy/image hierarchy is mandatory
- final prompt must describe a finished social asset, not an empty visual waiting for design later
- exact text or strongly implied copy structure should be compositionally integrated

### Base Visual Rules
If delivery_mode = base_visual:
- do not accidentally drift into finished-poster instructions
- allow clean image-led treatment
- avoid pseudo-poster language

## Poster Grade Requirement
For:
- project_launch
- construction_update
- festival_post
- amenity_spotlight
- site_visit_invite
- testimonial
- ad

the default expectation is a finished poster-grade output.

A poster-grade output must:
- choose a style family
- choose clear style primitives / levers
- show one dominant idea
- have visible hierarchy at first glance
- use the project/image as part of a designed composition, not as a plain background
- include a graphic-design layer or strong compositional device
- feel like a finished social asset, not an intermediate render

Reject outputs that feel like:
- plain hero image
- scenic architectural render
- building photo plus generic text
- nice image with some copy
- same template with different project names

## Strategic Framing by Post Type

### project_launch
- business_job: introduce the project, create desire, establish identity, create memorability
- persuasion_modes: aspiration, prestige, exclusivity, optimism, serenity, address_pride, quiet_luxury
- mistakes:
  - township explainer feel
  - brochure page feel
  - too many equal subjects
  - feature-list-as-idea
  - scenic render mistaken for a finished launch creative
  - repeated launch patterns regardless of project fit

### construction_update
- business_job: prove progress, build trust, show motion without exaggeration
- persuasion_modes: trust, credibility, transparency, confidence
- mistakes:
  - launch-ad feel
  - polishing beyond truth
  - hiding actual state
  - construction photo with simple caption instead of a designed update poster

### festival_post
- business_job: maintain branded presence, build warmth, stay culturally relevant
- persuasion_modes: warmth, elegance, ceremony, calm delight
- mistakes:
  - greeting-card clutter
  - Canva feel
  - forcing project imagery where symbolic treatment is stronger
  - decorative noise without hierarchy

### testimonial
- business_job: build trust, humanize the project, convert social proof into desirability
- persuasion_modes: trust, reassurance, warmth, quiet confidence
- mistakes:
  - generic quote card
  - stock-family cliché
  - text-heavy flyer feel
  - weak editorial hierarchy

### ad
- business_job: stop scroll, communicate one clear value proposition, drive inquiry without killing brand
- persuasion_modes: clarity, urgency, exclusivity, aspiration, curiosity
- mistakes:
  - cheap lead-gen styling
  - too many offers
  - pretty image but no usable ad hierarchy
  - generic promo poster feel

### site_visit_invite
- business_job: show real on-ground presence, create credibility and human connection
- persuasion_modes: authenticity, activity, trust, engagement
- mistakes:
  - event album feel
  - plain candid image with weak label treatment
  - generic corporate snapshot styling

### amenity_spotlight
- business_job: highlight one specific amenity as a lifestyle benefit
- persuasion_modes: lifestyle_desire, aspiration, family_warmth, quiet_luxury, community
- mistakes:
  - feature-list behavior
  - hiding the amenity behind building hero
  - plain amenity image with normal title only

## Required Output Contract
Return structured decisions for:
- post_type
- business_job
- persuasion_modes
- delivery_mode
- poster_archetype
- style_modifiers
- hero_presentation
- layout_geometry
- graphic_layer
- type_voice
- text_architecture
- mood_mode
- density
- brand_visibility
- truth_to_preserve
- asset_strategy
- required_data
- forbidden_moves
- fallback_plan
- final_prompt

## Universal Failure Modes
Revise if:
- wrong business job
- wrong persuasion mode
- weak hero
- genericized architecture
- no obvious hierarchy
- too many ideas competing
- brochure-spread feel
- fake premium tone
- device overwhelms project
- footer destroys readability
- stale repeated pattern
- weak fallback handling
- plain hero image + ordinary text behavior
- output feels like a base visual instead of a finished social asset
"""

# ============================================================
# SKILL 2: briefly-social-archetypes
# ============================================================
briefly_social_archetypes_md = r"""
---
name: briefly-social-archetypes
description: Style-first system for Briefly Social. Defines style primitives, style families, modifiers, compatibility rules, and anti-repetition rules.
metadata:
  version: "5.1.0"
  tags: ["real-estate", "social-media", "style-system", "archetypes", "design-levers"]
---

# Briefly Social Style System

Use this skill after the communication job is clear.

## Core Principle
A post type is not a design style.
Choose style based on:
- communication job
- asset fit
- payload fit
- market tone
- desired distinctiveness

## Style Primitives / Levers

### hero_presentation
How the visual subject appears.
Examples:
- single_tower
- villa
- township_overview
- facade_crop
- cutout_object
- monolith_icon
- entrance_arrival
- framed_image_card
- candid_presence
- amenity_hero
- symbolic_centerpiece

### layout_geometry
How the poster is structurally arranged.
Examples:
- centered_symmetry
- left_copy_right_hero
- right_copy_left_hero
- claim_panel_side_crop
- inset_card
- billboard_headline_sky
- footer_strip
- open_editorial_field
- framed_catalog
- split_panel
- lower_hero_upper_copy
- swiss_grid
- documentary_crop_overlay
- balanced_card_layout

### graphic_layer
What design layer supports the poster.
Examples:
- none
- organic_shape
- geometric_blocks
- line_art_watermark
- brand_watermark
- thin_frame
- translucent_panel
- proposition_box
- image_card
- architectural_tracing
- soft_gradient_field
- paper_depth
- divider_lines

### type_voice
How the typography should feel.
Examples:
- modern_sans
- premium_serif
- serif_sans_mix
- condensed_statement
- swiss_clean
- builder_readable
- fashion_editorial
- quiet_premium

### text_architecture
How the copy is organized.
Examples:
- slogan_first
- proposition_first
- scarcity_first
- emotional_headline_first
- philosophy_first
- address_first
- configuration_first
- quote_first
- footer_heavy
- one_statement

### mood_mode
Examples:
- crisp_daylight
- pale_editorial_daylight
- soft_morning
- golden_hour_optimism
- dusk_luxury
- twilight_calm
- ivory_studio_neutral
- warm_muted_premium
- cool_sustainable_daylight

### density
Examples:
- ultra_lean
- lean
- medium
- heavy
- regulation_heavy

### brand_visibility
Examples:
- whisper
- elegant_signature
- visible_brand_led
- campaign_dominant
- logo_forward
- developer_explicit

## Style Families

### soft_editorial_cutout
- best_for: approachable premium launch, builder-friendly premium, amenity-led clean statements
- default levers:
  - hero_presentation: cutout_object
  - layout_geometry: open_editorial_field
  - graphic_layer: organic_shape or line_art_watermark
  - type_voice: serif_sans_mix or quiet_premium
  - text_architecture: slogan_first or proposition_first
  - mood_mode: warm_muted_premium or pale_editorial_daylight
  - density: lean or medium
  - brand_visibility: elegant_signature

### centered_monolith
- best_for: one iconic tower, strong single-hero presence
- default levers:
  - hero_presentation: monolith_icon
  - layout_geometry: centered_symmetry
  - graphic_layer: geometric_blocks or none
  - type_voice: premium_serif or serif_sans_mix
  - text_architecture: proposition_first or scarcity_first
  - mood_mode: ivory_studio_neutral or pale_editorial_daylight
  - density: ultra_lean or lean
  - brand_visibility: visible_brand_led

### scarcity_panel
- best_for: exclusivity, limited inventory, ad-grade clarity
- default levers:
  - hero_presentation: single_tower
  - layout_geometry: claim_panel_side_crop
  - graphic_layer: proposition_box or geometric_blocks
  - type_voice: modern_sans or condensed_statement
  - text_architecture: scarcity_first
  - mood_mode: crisp_daylight or ivory_studio_neutral
  - density: lean or medium
  - brand_visibility: campaign_dominant

### side_crop_premium_tower
- best_for: strong architecture, readable launch/ad hybrid
- default levers:
  - hero_presentation: single_tower or facade_crop
  - layout_geometry: left_copy_right_hero or right_copy_left_hero
  - graphic_layer: none or thin_frame or translucent_panel
  - type_voice: modern_sans or serif_sans_mix
  - text_architecture: slogan_first or proposition_first
  - mood_mode: crisp_daylight, golden_hour_optimism, or dusk_luxury
  - density: lean or medium
  - brand_visibility: visible_brand_led

### philosophy_open_field
- best_for: worldview-led copy, sustainability, design intelligence
- default levers:
  - hero_presentation: architecture_with_environment
  - layout_geometry: open_editorial_field
  - graphic_layer: architectural_tracing or soft_gradient_field
  - type_voice: premium_serif or fashion_editorial
  - text_architecture: philosophy_first
  - mood_mode: pale_editorial_daylight or cool_sustainable_daylight
  - density: lean
  - brand_visibility: whisper or elegant_signature

### dusk_emotional_crop
- best_for: mood-led premium launch, poetic premium, aspirational evening emotion
- default levers:
  - hero_presentation: tower_with_dusk
  - layout_geometry: left_copy_right_hero or right_copy_left_hero
  - graphic_layer: soft_gradient_field or thin_frame
  - type_voice: premium_serif or serif_sans_mix
  - text_architecture: emotional_headline_first
  - mood_mode: dusk_luxury or twilight_calm
  - density: lean
  - brand_visibility: elegant_signature

### clear_sky_statement
- best_for: sharp, readable, in-feed direct launch ads
- default levers:
  - hero_presentation: tower_with_sky
  - layout_geometry: billboard_headline_sky
  - graphic_layer: none or thin_frame
  - type_voice: modern_sans or condensed_statement
  - text_architecture: proposition_first
  - mood_mode: crisp_daylight or golden_hour_optimism
  - density: lean or medium
  - brand_visibility: visible_brand_led

### footer_builder_campaign
- best_for: market-facing builder creatives, heavier payload, practical shareable campaigns
- default levers:
  - hero_presentation: single_tower or entrance_arrival
  - layout_geometry: footer_strip or lower_hero_upper_copy
  - graphic_layer: color_band or divider_lines
  - type_voice: builder_readable or modern_sans
  - text_architecture: proposition_first or footer_heavy
  - mood_mode: crisp_daylight or golden_hour_optimism
  - density: medium or heavy
  - brand_visibility: developer_explicit

### white_space_editorial_statement
- best_for: restrained premium, boutique luxury, concept-led minimalism
- default levers:
  - hero_presentation: cutout_object or monolith_icon
  - layout_geometry: open_editorial_field or centered_symmetry
  - graphic_layer: thin_frame or architectural_tracing or none
  - type_voice: premium_serif or swiss_clean
  - text_architecture: one_statement or address_first
  - mood_mode: ivory_studio_neutral or pale_editorial_daylight
  - density: ultra_lean
  - brand_visibility: whisper or elegant_signature

### masterplan_scale_reveal
- best_for: township scale, masterplan-led launches where scale is truly the message
- default levers:
  - hero_presentation: township_overview or aerial_masterplan
  - layout_geometry: balanced_card_layout or framed_catalog
  - graphic_layer: translucent_panel or thin_frame
  - type_voice: modern_sans or builder_readable
  - text_architecture: proposition_first
  - mood_mode: golden_hour_optimism or crisp_daylight
  - density: medium
  - brand_visibility: visible_brand_led

### documentary_presence
- best_for: site visits, construction updates, candid credibility
- default levers:
  - hero_presentation: candid_presence
  - layout_geometry: documentary_crop_overlay
  - graphic_layer: translucent_panel or divider_lines
  - type_voice: builder_readable or modern_sans
  - text_architecture: proposition_first
  - mood_mode: crisp_daylight or warm_muted_premium
  - density: medium
  - brand_visibility: developer_explicit

### quote_led_editorial
- best_for: testimonial, trust, quote-first social proof
- default levers:
  - hero_presentation: framed_image_card or architecture_with_environment
  - layout_geometry: balanced_card_layout or open_editorial_field
  - graphic_layer: thin_frame or image_card
  - type_voice: premium_serif or quiet_premium
  - text_architecture: quote_first
  - mood_mode: warm_muted_premium
  - density: lean or medium
  - brand_visibility: elegant_signature

### symbolic_festive_field
- best_for: festival posts where symbolism is stronger than property display
- default levers:
  - hero_presentation: symbolic_centerpiece
  - layout_geometry: centered_symmetry or open_editorial_field
  - graphic_layer: soft_gradient_field or architectural_tracing
  - type_voice: premium_serif or quiet_premium
  - text_architecture: one_statement
  - mood_mode: warm_muted_premium or twilight_calm
  - density: ultra_lean or lean
  - brand_visibility: elegant_signature

### organic_shape_launch
- best_for: approachable premium, family aspirational, builder-friendly distinctiveness
- default levers:
  - hero_presentation: cutout_object or midrise_block
  - layout_geometry: lower_hero_upper_copy or split_panel
  - graphic_layer: organic_shape
  - type_voice: modern_sans or serif_sans_mix
  - text_architecture: slogan_first or proposition_first
  - mood_mode: warm_muted_premium or soft_morning
  - density: medium
  - brand_visibility: visible_brand_led

### watermark_catalog
- best_for: catalog-like premium, villas, quiet luxury, interior/villa image-card
- default levers:
  - hero_presentation: framed_image_card
  - layout_geometry: framed_catalog
  - graphic_layer: brand_watermark or image_card
  - type_voice: premium_serif
  - text_architecture: emotional_headline_first or address_first
  - mood_mode: ivory_studio_neutral or warm_muted_premium
  - density: lean
  - brand_visibility: whisper

### inset_image_card
- best_for: catalog page treatment, villa or amenity posters, soft premium
- default levers:
  - hero_presentation: framed_image_card
  - layout_geometry: inset_card
  - graphic_layer: image_card or thin_frame
  - type_voice: quiet_premium or premium_serif
  - text_architecture: emotional_headline_first
  - mood_mode: warm_muted_premium
  - density: lean
  - brand_visibility: elegant_signature

### swiss_grid_premium
- best_for: sharp modern premium, disciplined address-led or proposition-led communication
- default levers:
  - hero_presentation: monolith_icon or facade_crop
  - layout_geometry: swiss_grid
  - graphic_layer: thin_frame or geometric_blocks
  - type_voice: swiss_clean
  - text_architecture: address_first or proposition_first
  - mood_mode: ivory_studio_neutral
  - density: ultra_lean or lean
  - brand_visibility: visible_brand_led

### ultra_minimal_address
- best_for: boutique luxury, address prestige, one-line statement
- default levers:
  - hero_presentation: cutout_object or monolith_icon
  - layout_geometry: centered_symmetry or open_editorial_field
  - graphic_layer: none or thin_frame
  - type_voice: premium_serif
  - text_architecture: address_first or one_statement
  - mood_mode: ivory_studio_neutral
  - density: ultra_lean
  - brand_visibility: whisper

## Style Modifiers
Use 0-2 modifiers only. They adapt the base style without changing the underlying family.
Examples:
- scarcity
- dusk
- festive
- qr_ready
- compliance_footer
- family_tone
- eco_tone
- urban_tone
- boutique_luxury
- campaign_dramatic
- calm_editorial
- ad_performance
- poetic
- address_prestige

## Compatibility Rules

### Core
1. Style family must fit the asset, not just the mood.
2. text_architecture must support the persuasion mode.
3. graphic_layer must not overpower the hero.
4. density must fit the payload.
5. Do not pick a style family only because it looks nice; pick it because it suits the job and asset.

### External Input Rules
1. If a reference image exists, first decide what the strongest hero actually is before choosing style family.
2. If a reference image contains many buildings, do not assume all of them should stay equally important.
3. Template image may guide composition, graphic layer, and finish, but never identity.
4. Logo image affects brand visibility and placement discipline, not hero selection.
5. If template cue and reference truth conflict, reference truth wins.

### Strong Fits
- single_tower → centered_monolith, side_crop_premium_tower, scarcity_panel, clear_sky_statement, dusk_emotional_crop, swiss_grid_premium
- villa → watermark_catalog, inset_image_card, white_space_editorial_statement
- township_overview → masterplan_scale_reveal
- candid_presence → documentary_presence
- amenity_hero → soft_editorial_cutout, inset_image_card, watermark_catalog
- symbolic_centerpiece → symbolic_festive_field

### Weak / Invalid Fits
- township_overview + centered_monolith
- township_overview + scarcity_panel
- testimonial + giant launch slogan behavior
- site_visit_invite + monolithic luxury tower treatment
- construction_update + dusk fantasy polish
- villa + one_tower claim
- heavy regulatory payload + ultra_minimal_address unless a compliance footer is explicitly added
- any finished poster + plain scenic render logic

## Anti-Repetition Rules
Revise if:
- outputs look swappable across projects
- only color changes but structure stays the same
- every launch uses right tower + left text
- every premium poster is white background + centered tower
- every ad becomes scarcity-led
- style family is chosen but not visibly expressed in the final prompt
"""

# ============================================================
# SKILL 3: briefly-social-qa
# ============================================================
briefly_social_qa_md = r"""
---
name: briefly-social-qa
description: Final QA skill covering business-job fit, style-family fidelity, lever coherence, asset grounding, hierarchy, external-input handling, and anti-repetition checks.
metadata:
  version: "5.1.0"
  tags: ["real-estate", "social-media", "qa", "style-fidelity", "asset-truth", "hierarchy"]
---

# Briefly Social QA

Use this skill for final critique before output.

## Part A — Job / Style Separation Check
Always ask:
1. Is the communication job clear?
2. Is the style family clear?
3. Are they correctly matched?
4. Is the result repeating a default layout instead of using the chosen style system?

A post may be correct in post type and still fail because the style behavior is stale, generic, or mismatched.

## Part B — Copy / Visual Hierarchy

### Headline Guidance
- one_statement: 2-6 words, poster-like, visually dominant
- proposition_first: one big idea + one short support line
- scarcity_first: one scarcity claim dominates, no feature-list clutter
- emotional_headline_first: softer, slightly longer, calmer imagery
- philosophy_first: headline + short paragraph, needs breathing room
- quote_first: quote is the hero, not the building

### Density
- ultra_lean: brand + statement + optional small footer
- lean: brand + project + headline + one support
- medium: + configuration/location/contact
- heavy / regulation_heavy: structured footer, subordinate to main headline

### Footer Rules
Footer supports, never dominates.
RERA/contact/QR must stay subordinate unless brief demands compliance emphasis.

### Placement
- side-crop posters need a designed copy field, not random empty sky
- monolith posters need headline/tower separation
- inset-card posters need clear outer structure
- dusk posters need lighter, calmer copy behavior
- finished posters must integrate text into composition rather than leaving generic room for later text

## Part C — Asset Grounding

### Core Rules
1. Never invent filenames.
2. Never use an unavailable asset as if it exists.
3. Prefer one strong primary anchor over many weak anchors.
4. Preserve actual architecture, amenity design, construction state, and site context.
5. If style family requires different asset behavior than available, change style family first.
6. If delivery_mode is finished_poster, do not fall back to scenic hero-image logic.

### External Input Checks
1. If a reference image exists, the project must remain recognisable relative to that image.
2. Safe improvements are allowed: crop, angle refinement, realism polish, hierarchy, atmosphere.
3. Unsafe changes are not allowed: altered facade identity, altered tower count, altered construction stage, invented amenity geometry.
4. If a template image exists, check that it shaped style only, not identity.
5. If a logo image exists, check that the logo is treated as exact, not stylized or rewritten.

### Asset Tiers
- T1: exact project anchor
- T2: same-project support
- T3: same-brand contextual
- T4: symbolic/graphic-led
- T5: forbidden substitute

### Post-Type Asset Rules
- project_launch: prefer T1; use T2 or T4 only when strategy allows
- construction_update: require T1 real progress
- festival_post: T4 symbolic often strongest
- testimonial: authentic support or T4 editorial background
- ad: strongest desirability from T1/T2
- site_visit_invite: require real credibility if visit is implied
- amenity_spotlight: exact amenity first

### Fallback Logic
- no strong launch hero → shift style family before inventing detail
- weak side-crop asset → centered_monolith or cutout_object treatment
- truthful asset visually weak → improve through crop, hierarchy, mood, and device; never through invention

## Part D — Style Family Fidelity
Check whether the final prompt visibly behaves like the selected style family.

Examples:
- soft_editorial_cutout should show a cutout/object behavior plus calm field plus graphic softness
- centered_monolith should show central icon presence and sparse hierarchy
- scarcity_panel should show a claim block and strong exclusivity structure
- watermark_catalog should feel like a catalog page, not an ad panel
- swiss_grid_premium should show alignment discipline, not blob-led softness

Revise if the style family is only named but not manifested.

## Part E — Quality Critic

Critique in this order:

### 1) Business Job
Is the job obvious at a glance?

### 2) Hero
Is there a clear hero?
Is it actually dominant?

### 3) Hierarchy
What is read first / second / third?
Is text architecture suited to payload?

### 4) Style Coherence
Do hero, layout, graphic layer, type voice, mood, and density feel like one family?

### 5) Freshness
Does the prompt avoid default repeated patterns?
Would it produce something distinct from recent outputs?

### 6) Market Fit
Too editorial for the developer?
Too builder-ish for boutique?
Too experimental for weak assets?

### 7) Truth
Project still recognisable?
Any false completion, false amenity, or false scale implied?

## Revision Triggers
Revise if:
- no singular dominant idea
- belongs to any random project
- genericized architecture
- copy too long for chosen structure
- repetitive composition
- brochure / flyer / Canva feel
- premium only in words, not in structure
- result feels like a plain hero image with normal text
- composition has no real poster logic
- text appears simply placed onto a render
- there is no meaningful graphic or editorial design layer
- output feels like a base visual rather than a finished asset
- chosen style family is not visibly executed
- template image is likely to override project identity
- logo could be distorted or rewritten

## Final Approval Standard
Approve only if:
- clear
- on-strategy
- recognisable
- style-distinct
- immediately usable
- unmistakably poster-grade rather than base-visual-grade
"""

style_primitive_matrix_md = r"""
# Style Primitive Matrix

| lever | purpose | choose_how_many |
|---|---|---:|
| persuasion_modes | what the post should make the viewer feel/believe | 1-2 |
| hero_presentation | how the main subject appears | 1 |
| layout_geometry | structural composition logic | 1 |
| graphic_layer | design device language | 0-2 |
| type_voice | typography personality | 1 |
| text_architecture | text hierarchy behavior | 1 |
| mood_mode | light / atmosphere / emotional climate | 1 |
| density | information load | 1 |
| brand_visibility | how loudly the brand appears | 1 |
| style_modifiers | optional adaptation layer | 0-2 |
"""

style_family_map_md = r"""
# Style Family Map

| style_family | strongest_for | typical hero | typical text behavior | core risk |
|---|---|---|---|---|
| soft_editorial_cutout | approachable premium launch | cutout object | slogan / proposition | generic blob template |
| centered_monolith | one-tower statement | monolith icon | proposition / scarcity | weak tiny hero |
| scarcity_panel | boutique exclusivity | side-crop tower | scarcity first | panel overwhelms hero |
| side_crop_premium_tower | premium tower launch | tall tower crop | slogan / proposition | generic luxury ad |
| philosophy_open_field | worldview-led copy | architecture + field | philosophy first | weak conversion |
| dusk_emotional_crop | mood-led premium launch | dusk tower | emotional | generic mood render |
| clear_sky_statement | direct readable launch | tower with sky | proposition first | flat default ad |
| footer_builder_campaign | info-heavy builder launch | tower / arrival | proposition + footer | clutter |
| white_space_editorial_statement | restrained boutique premium | monolith / cutout | one statement | empty render feel |
| masterplan_scale_reveal | township launch | township overview | proposition | brochure drift |
| documentary_presence | site visit / construction | candid presence | proposition | weak event-album feel |
| quote_led_editorial | testimonial | card / support image | quote first | bland quote card |
| symbolic_festive_field | festival | symbolic centerpiece | one statement | greeting-card clutter |
| organic_shape_launch | approachable premium | cutout / mid-rise | slogan / proposition | overdesigned friendliness |
| watermark_catalog | catalog-like premium | framed image card | emotional / address | too passive |
| inset_image_card | card-led premium | framed image card | emotional | weak poster presence |
| swiss_grid_premium | sharp modern premium | monolith / facade crop | address / proposition | too clinical |
| ultra_minimal_address | address prestige | monolith / cutout | address first | too sparse |
"""

# Write consolidated skills
write_text(SKILLS_DIR / "briefly-social-core" / "SKILL.md", briefly_social_core_md)
write_text(SKILLS_DIR / "briefly-social-archetypes" / "SKILL.md", briefly_social_archetypes_md)
write_text(SKILLS_DIR / "briefly-social-qa" / "SKILL.md", briefly_social_qa_md)

write_text(
    SKILLS_DIR / "briefly-social-core" / "references" / "style-primitive-matrix.md",
    style_primitive_matrix_md,
)
write_text(
    SKILLS_DIR / "briefly-social-archetypes" / "references" / "style-family-map.md",
    style_family_map_md,
)

# Clean up old redundant skill directories if they exist
import shutil
for old in [
    "briefly-social-strategic-framing",
    "briefly-social-creative-levers",
    "briefly-social-poster-archetypes",
    "briefly-social-combination-rules",
    "briefly-social-copy-visual-balance",
    "briefly-social-asset-grounding",
    "briefly-social-quality-critic",
]:
    old_path = SKILLS_DIR / old
    if old_path.exists():
        shutil.rmtree(old_path)

## 5. Data

In [165]:
brand_guidelines: Dict[str, Any] = {
    "brand_name": "Horizon Living",
    "tagline": "Elevate Your Everyday",
    "visual_positioning": "Premium editorial real-estate brand with warm, clean, uncluttered social visuals",
    "architecture_style_brand_note": "Contemporary modern presentation; do not override project-specific architecture",
    "primary_colors": ["#1A3C5E", "#C8A96E"],
    "secondary_colors": ["#F5F1EB", "#2E2E2E"],
    "typography_mood": "Minimal, modern, premium sans-serif",
    "target_audience": "Urban Indian families, age 28-45, upper-middle to premium segment",
    "people_representation": "Natural Indian families or professionals, warm and believable, not stock-photo stiff",
    "text_policy_default": "reserve_space",
    "visual_donts": [
        "Do not invent logos",
        "Do not force heavy text into the generated image unless explicitly requested",
        "Do not use cartoon or illustrated style",
        "Do not use neon or oversaturated colors",
        "Do not use cluttered brochure-style layouts",
        "Do not let brand styling override project truth"
    ],
    "default_aspect_ratios": {
        "social_post": "1:1",
        "story": "9:16",
        "banner": "16:9"
    }
}

projects: Dict[str, Any] = {
    "horizon-heights": {
        "project_name": "Horizon Heights",
        "location": "Whitefield, Bangalore",
        "type": "Premium Residential Tower",
        "floors": 32,
        "units": "2 BHK, 3 BHK, 4 BHK",
        "status": "Under Construction — 18 floors completed",
        "completion_date": "Q4 2027",
        "usp": "Rooftop infinity pool, sky lounge, EV charging on every floor",
        "architecture_description": "Sleek glass-and-steel tower with a curved facade, vertical garden strips on alternating floors, grand double-height entrance lobby with water feature",
        "rera_number": "PRM/KA/RERA/1251/446/PR/220101/004567",
        "contact": "+91 98765 43210",
        "amenities": [
            "Infinity swimming pool (rooftop)",
            "Sky lounge (32nd floor)",
            "Fully equipped gymnasium",
            "Landscaped central garden with jogging track",
            "Kids' adventure play zone",
            "Co-working lounge",
            "Mini theatre",
            "EV charging stations"
        ],
        "brand_assets": {
            "logo_primary": {
                "available": True,
                "filename": "HH_LOGO_PRIMARY.png",
                "description": "Primary Horizon Heights logo lockup on transparent background"
            }
        },
        "image_library": {
            "main_building_exterior": {
                "available": True,
                "filename": "HH_EXTERIOR_01.png",
                "description": "Front elevation render of the 32-story tower at twilight, glass facade lit up, entrance lobby visible"
            },
            "aerial_view": {
                "available": True,
                "filename": "HH_AERIAL_01.png",
                "description": "Drone shot from 200m showing the tower, surrounding landscaping, and nearby tech park"
            },
            "construction_progress": {
                "available": True,
                "filename": "HH_CONSTRUCTION_01.png",
                "description": "Site photo showing 18 completed floors, crane on top, blue safety nets, workers visible on upper deck"
            },
            "entrance_gate": {
                "available": True,
                "filename": "HH_ENTRANCE_01.png",
                "description": "Grand double-height entrance lobby with water feature and landscaped driveway"
            },
            "swimming_pool": {
                "available": True,
                "filename": "HH_POOL_01.png",
                "description": "Rooftop infinity pool render with city skyline in background, sun loungers, potted palms, dusk lighting"
            },
            "sky_lounge": {
                "available": True,
                "filename": "HH_SKYLOUNGE_01.png",
                "description": "32nd floor sky lounge interior with floor-to-ceiling windows, city panorama, warm pendant lighting, leather seating"
            },
            "gymnasium": {
                "available": True,
                "filename": "HH_GYM_01.png",
                "description": "Modern gym interior with floor-to-ceiling windows overlooking the garden"
            },
            "sample_flat_living_room": {
                "available": True,
                "filename": "HH_LIVING_01.png",
                "description": "Spacious living room with large windows, modern furniture, warm lighting, city view"
            },
            "sales_gallery": {
                "available": True,
                "filename": "HH_SALES_01.png",
                "description": "Sales gallery interior with scale model of the tower, reception desk, brochure display"
            },
        }
    },
    "horizon-greens": {
        "project_name": "Horizon Greens",
        "location": "Sarjapur Road, Bangalore",
        "type": "Integrated Township",
        "floors": "G+14 (8 towers)",
        "units": "1 BHK, 2 BHK, 3 BHK",
        "status": "Newly Launched",
        "completion_date": "Q2 2029",
        "usp": "35 acres of green township with 70% open space, pet-friendly zones, organic community farm",
        "architecture_description": "Cluster of eight pastel-toned mid-rise towers arranged around a massive central park, each tower with a distinct earthy color accent, wide tree-lined internal roads",
        "rera_number": "PRM/KA/RERA/1251/446/PR/230715/006789",
        "contact": "+91 98765 12345",
        "amenities": [
            "Olympic-size swimming pool",
            "Clubhouse with banquet hall",
            "Cricket practice nets",
            "Yoga and meditation pavilion",
            "Cycling and jogging loop (1.2 km)",
            "Organic community farm",
            "Pet park with agility course",
            "Senior citizen garden with reflexology path"
        ],
        "brand_assets": {
            "logo_primary": {
                "available": True,
                "filename": "HG_LOGO_PRIMARY.png",
                "description": "Primary Horizon Greens logo lockup on transparent background"
            }
        },
        "image_library": {
            "main_building_exterior": {
                "available": True,
                "filename": "HG_TOWER_A_01.png",
                "description": "Front elevation of Tower A, pastel terracotta tone, balconies with planters, ground-floor retail visible"
            },
            "aerial_view": {
                "available": True,
                "filename": "HG_AERIAL_01.png",
                "description": "Drone shot of the full 35-acre township, all 8 towers visible, central park in the middle, tree-lined roads"
            },
            "township_overview": {
                "available": True,
                "filename": "HG_TOWNSHIP_01.png",
                "description": "Wide-angle ground-level view of the township entrance with multiple towers in background, landscaped boulevard"
            },
            "construction_progress": {
                "available": False,
                "filename": None,
                "description": None
            },
            "entrance_gate": {
                "available": True,
                "filename": "HG_ENTRANCE_01.png",
                "description": "Grand township entrance gate with security booth, wide road, avenue trees, welcome signage"
            },
            "swimming_pool": {
                "available": True,
                "filename": "HG_POOL_01.png",
                "description": "Olympic-size swimming pool with lane markers, poolside cabanas, clubhouse building in background"
            },
            "clubhouse": {
                "available": True,
                "filename": "HG_CLUB_01.png",
                "description": "Two-story clubhouse exterior with glass facade, banquet hall visible on upper floor, landscaped forecourt"
            },
            "garden_jogging_track": {
                "available": True,
                "filename": "HG_JOGTRACK_01.png",
                "description": "1.2 km cycling and jogging loop winding through landscaped gardens with distance markers"
            },
            "yoga_pavilion": {
                "available": True,
                "filename": "HG_YOGA_01.png",
                "description": "Open-air yoga pavilion with wooden deck, surrounded by bamboo groves, morning mist, calm meditative atmosphere"
            },
            "organic_farm": {
                "available": True,
                "filename": "HG_FARM_01.png",
                "description": "Community organic farm with raised beds, families gardening, small greenhouse in corner"
            },
            "pet_park": {
                "available": True,
                "filename": "HG_PETPARK_01.png",
                "description": "Fenced pet park with agility course obstacles, dogs playing, owners on benches"
            },
            "sample_flat_living_room": {
                "available": True,
                "filename": "HG_LIVING_01.png",
                "description": "Open-plan living and dining area with balcony overlooking the central park, earthy tones"
            },
            "sample_flat_bedroom": {
                "available": True,
                "filename": "HG_BEDROOM_01.png",
                "description": "Compact master bedroom with built-in wardrobe, large window with garden view"
            },
            "sample_flat_kitchen": {
                "available": True,
                "filename": "HG_KITCHEN_01.png",
                "description": "Modular kitchen with breakfast counter, pastel cabinetry, window above sink"
            },
            "sales_gallery": {
                "available": True,
                "filename": "HG_SALES_01.png",
                "description": "Sales gallery with large township scale model, interactive screens, lounge seating"
            }
        }
    }
}

write_text(DATA_DIR / "brand_guidelines.json", json.dumps(brand_guidelines, indent=2))
write_text(DATA_DIR / "projects.json", json.dumps(projects, indent=2))

In [166]:
# ------------------------------------------------------------------------------
# 5A) Runtime default overrides
# Keeps the existing data cell intact, but updates defaults for poster generation
# ------------------------------------------------------------------------------

brand_guidelines["text_policy_default"] = "exact_text"
brand_guidelines["delivery_mode_default"] = "finished_poster"
brand_guidelines["visual_donts"] = list(dict.fromkeys(
    brand_guidelines["visual_donts"] + [
        "Do not generate a plain hero image with normal text on top",
        "Do not treat empty sky or generic clean space as poster composition",
        "Do not produce scenic base visuals when a finished social poster is expected",
    ]
))

write_text(DATA_DIR / "brand_guidelines.json", json.dumps(brand_guidelines, indent=2))

## 6. Enums + Schemas

In [167]:
"""## 6. Enums + Schemas"""

class PostType(str, Enum):
    project_launch = "project_launch"
    construction_update = "construction_update"
    festival_post = "festival_post"
    amenity_spotlight = "amenity_spotlight"
    site_visit_invite = "site_visit_invite"


class FormatType(str, Enum):
    social_post = "social_post"
    story = "story"
    banner = "banner"


class VisualMode(str, Enum):
    asset_faithful = "asset_faithful"
    editorialized_truth = "editorialized_truth"
    graphic_led = "graphic_led"


class DeliveryMode(str, Enum):
    finished_poster = "finished_poster"
    base_visual = "base_visual"


class TextPolicy(str, Enum):
    none = "none"
    exact_text = "exact_text"


class StylizationLevel(str, Enum):
    low = "low"
    medium = "medium"
    medium_high = "medium_high"


# NOTE:
# This enum is now effectively the style-family enum.
# Kept under the same name for backward compatibility with the notebook.
class PosterArchetype(str, Enum):
    soft_editorial_cutout = "soft_editorial_cutout"
    centered_monolith = "centered_monolith"
    scarcity_panel = "scarcity_panel"
    side_crop_premium_tower = "side_crop_premium_tower"
    philosophy_open_field = "philosophy_open_field"
    dusk_emotional_crop = "dusk_emotional_crop"
    clear_sky_statement = "clear_sky_statement"
    footer_builder_campaign = "footer_builder_campaign"
    white_space_editorial_statement = "white_space_editorial_statement"
    masterplan_scale_reveal = "masterplan_scale_reveal"
    documentary_presence = "documentary_presence"
    quote_led_editorial = "quote_led_editorial"
    symbolic_festive_field = "symbolic_festive_field"
    organic_shape_launch = "organic_shape_launch"
    watermark_catalog = "watermark_catalog"
    inset_image_card = "inset_image_card"
    swiss_grid_premium = "swiss_grid_premium"
    ultra_minimal_address = "ultra_minimal_address"


class HeroPresentation(str, Enum):
    single_tower = "single_tower"
    tower_pair = "tower_pair"
    midrise_block = "midrise_block"
    villa = "villa"
    township_overview = "township_overview"
    facade_crop = "facade_crop"
    entrance_arrival = "entrance_arrival"
    aerial_masterplan = "aerial_masterplan"
    cutout_object = "cutout_object"
    monolith_icon = "monolith_icon"
    architecture_with_environment = "architecture_with_environment"
    tower_with_sky = "tower_with_sky"
    tower_with_dusk = "tower_with_dusk"
    framed_image_card = "framed_image_card"
    candid_presence = "candid_presence"
    amenity_hero = "amenity_hero"
    symbolic_centerpiece = "symbolic_centerpiece"


class LayoutGeometry(str, Enum):
    centered_symmetry = "centered_symmetry"
    left_copy_right_hero = "left_copy_right_hero"
    right_copy_left_hero = "right_copy_left_hero"
    claim_panel_side_crop = "claim_panel_side_crop"
    inset_card = "inset_card"
    billboard_headline_sky = "billboard_headline_sky"
    footer_strip = "footer_strip"
    open_editorial_field = "open_editorial_field"
    framed_catalog = "framed_catalog"
    split_panel = "split_panel"
    lower_hero_upper_copy = "lower_hero_upper_copy"
    swiss_grid = "swiss_grid"
    documentary_crop_overlay = "documentary_crop_overlay"
    balanced_card_layout = "balanced_card_layout"


class GraphicLayer(str, Enum):
    none = "none"
    organic_shape = "organic_shape"
    geometric_blocks = "geometric_blocks"
    line_art_watermark = "line_art_watermark"
    brand_watermark = "brand_watermark"
    thin_frame = "thin_frame"
    translucent_panel = "translucent_panel"
    proposition_box = "proposition_box"
    image_card = "image_card"
    architectural_tracing = "architectural_tracing"
    soft_gradient_field = "soft_gradient_field"
    color_band = "color_band"
    paper_depth = "paper_depth"
    divider_lines = "divider_lines"


class TypeVoice(str, Enum):
    modern_sans = "modern_sans"
    premium_serif = "premium_serif"
    serif_sans_mix = "serif_sans_mix"
    condensed_statement = "condensed_statement"
    swiss_clean = "swiss_clean"
    builder_readable = "builder_readable"
    fashion_editorial = "fashion_editorial"
    quiet_premium = "quiet_premium"


class TextArchitecture(str, Enum):
    slogan_first = "slogan_first"
    proposition_first = "proposition_first"
    scarcity_first = "scarcity_first"
    emotional_headline_first = "emotional_headline_first"
    philosophy_first = "philosophy_first"
    address_first = "address_first"
    configuration_first = "configuration_first"
    quote_first = "quote_first"
    footer_heavy = "footer_heavy"
    one_statement = "one_statement"


class MoodMode(str, Enum):
    crisp_daylight = "crisp_daylight"
    pale_editorial_daylight = "pale_editorial_daylight"
    soft_morning = "soft_morning"
    golden_hour_optimism = "golden_hour_optimism"
    dusk_luxury = "dusk_luxury"
    twilight_calm = "twilight_calm"
    ivory_studio_neutral = "ivory_studio_neutral"
    warm_muted_premium = "warm_muted_premium"
    cool_sustainable_daylight = "cool_sustainable_daylight"


class DensityLevel(str, Enum):
    ultra_lean = "ultra_lean"
    lean = "lean"
    medium = "medium"
    heavy = "heavy"
    regulation_heavy = "regulation_heavy"


class BrandVisibility(str, Enum):
    whisper = "whisper"
    elegant_signature = "elegant_signature"
    visible_brand_led = "visible_brand_led"
    campaign_dominant = "campaign_dominant"
    logo_forward = "logo_forward"
    developer_explicit = "developer_explicit"


class AssetDecision(BaseModel):
    category: Optional[str] = Field(None)
    filename: Optional[str] = Field(None)
    reason: str = Field(...)
    reference_tag: Optional[str] = Field(None)
    fallback_categories: List[str] = Field(default_factory=list)
    recommended_poster_archetype: Optional[str] = Field(
        None,
        description="Recommended style family based on the chosen asset."
    )


class BriefAnalysis(BaseModel):
    project_slug: str = Field(..., description="Canonical project slug.")
    project_name: str = Field(..., description="Display name of the project.")
    post_type: PostType
    business_job: str = Field(..., description="One-line communication job for the post.")
    persuasion_modes: List[str] = Field(default_factory=list, description="1-2 persuasion modes.")
    style_modifiers: List[str] = Field(default_factory=list, description="Optional modifiers like scarcity, dusk, family_tone.")
    format_type: FormatType
    aspect_ratio: str = Field(..., description="Resolved aspect ratio, such as 1:1 or 9:16.")
    occasion: Optional[str] = Field(None, description="Festival or occasion if relevant.")
    specific_amenity: Optional[str] = Field(None, description="Exact amenity key if relevant.")
    no_building_image: bool = Field(..., description="Whether the user explicitly said to avoid showing the building.")
    visual_mode: VisualMode
    delivery_mode: DeliveryMode
    text_policy: TextPolicy
    poster_archetype: PosterArchetype = Field(..., description="Chosen style family for the poster.")
    hero_presentation: HeroPresentation
    layout_geometry: LayoutGeometry
    graphic_layer: List[GraphicLayer] = Field(default_factory=list, description="0-2 graphic layers.")
    type_voice: TypeVoice
    text_architecture: TextArchitecture
    mood_mode: MoodMode
    density: DensityLevel
    brand_visibility: BrandVisibility
    stylization_level: StylizationLevel
    truth_priority: str = Field(..., description="How strict factual fidelity must be.")
    objective_summary: str = Field(..., description="One-line summary of the post objective.")
    brand_summary: str = Field(..., description="Condensed brand guidance relevant to this post.")
    project_truth_summary: str = Field(..., description="Condensed factual project information relevant to this post.")
    asset_decision: AssetDecision = Field(...)
    required_data: List[str] = Field(default_factory=list, description="Required payload elements such as brand, project, headline, location, RERA, contact.")
    conflict_notes: List[str] = Field(default_factory=list, description="Any brief conflicts and how they were resolved.")
    logo_asset_filename: Optional[str] = Field(None, description="Exact logo file if available for post-processing or planning.")
    negative_rules: List[str] = Field(default_factory=list, description="Critical negative constraints to preserve truth.")


class CraftedPrompt(BaseModel):
    post_type: PostType
    business_job: str
    project_name: str
    format_type: FormatType
    aspect_ratio: str
    visual_mode: VisualMode
    delivery_mode: DeliveryMode
    text_policy: TextPolicy
    poster_archetype: PosterArchetype
    style_modifiers: List[str] = Field(default_factory=list)
    hero_presentation: HeroPresentation
    layout_geometry: LayoutGeometry
    graphic_layer: List[GraphicLayer] = Field(default_factory=list)
    type_voice: TypeVoice
    text_architecture: TextArchitecture
    mood_mode: MoodMode
    density: DensityLevel
    brand_visibility: BrandVisibility
    primary_reference_filename: Optional[str] = None
    prompt: str = Field(..., description="Final image generation prompt.")
    negative: str = Field(..., description="Compact negative string.")
    rationale: str = Field(..., description="Short explanation of why this prompt structure works.")


class VerificationResult(BaseModel):
    approved: bool = Field(..., description="Whether the final prompt passed verification.")
    issues: List[str] = Field(default_factory=list, description="Detected issues.")
    revised_prompt: str = Field(..., description="Verified or revised prompt.")
    revised_negative: str = Field(..., description="Verified or revised negative constraints.")
    verification_summary: str = Field(..., description="Short summary of what was checked.")

In [168]:
"""## 6A. Analyst input + slimmed analysis schema patch"""

class AnalystInput(BaseModel):
    project_slug: str = Field(..., description="Canonical project slug supplied by orchestration.")
    user_brief: str = Field(..., description="Raw user brief for the social creative.")
    selected_post_type: Optional[PostType] = Field(
        None,
        description="Explicit post type selected by the caller. If provided, use this instead of inferring from the brief."
    )
    reference_image_paths: List[str] = Field(
        default_factory=list,
        description="Uploaded reference image file paths. These are identity / scene anchors."
    )
    reference_image_note: Optional[str] = Field(
        None,
        description="Optional instruction for how the reference images should be used."
    )
    template_id: Optional[str] = Field(
        None,
        description="Optional structured template id from the template library."
    )
    template_image_path: Optional[str] = Field(
        None,
        description="Optional template image path used as a style / composition cue."
    )
    template_note: Optional[str] = Field(
        None,
        description="Optional note about how strongly to follow the template."
    )
    logo_image_path: Optional[str] = Field(
        None,
        description="Optional logo image path. If provided, treat it as an exact brand mark."
    )
    logo_note: Optional[str] = Field(
        None,
        description="Optional note about logo usage."
    )


class AssetDecision(BaseModel):
    source: str = Field(
        ...,
        description="One of: project_library, uploaded_reference, none."
    )
    category: Optional[str] = Field(None)
    filename: Optional[str] = Field(None)
    filepath: Optional[str] = Field(None)
    reason: str = Field(...)
    reference_tag: Optional[str] = Field(None)
    fallback_categories: List[str] = Field(default_factory=list)
    recommended_poster_archetype: Optional[str] = Field(None)


class BriefAnalysis(BaseModel):
    project_slug: str = Field(..., description="Canonical project slug.")
    project_name: str = Field(..., description="Display name of the project.")

    post_type: PostType
    business_job: str = Field(..., description="One-line communication job for the post.")
    persuasion_modes: List[str] = Field(default_factory=list, description="1-2 persuasion modes.")
    style_modifiers: List[str] = Field(default_factory=list, description="Optional modifiers like scarcity, dusk, family_tone.")

    format_type: FormatType
    aspect_ratio: str = Field(..., description="Resolved aspect ratio, such as 1:1 or 9:16.")
    occasion: Optional[str] = Field(None, description="Festival or occasion if relevant.")
    specific_amenity: Optional[str] = Field(None, description="Exact amenity key if relevant.")
    no_building_image: bool = Field(..., description="Whether the brief explicitly avoids building imagery.")

    visual_mode: VisualMode
    delivery_mode: DeliveryMode
    text_policy: TextPolicy

    poster_archetype: PosterArchetype
    hero_presentation: HeroPresentation
    layout_geometry: LayoutGeometry
    graphic_layer: List[GraphicLayer] = Field(default_factory=list, description="0-2 graphic layers.")
    type_voice: TypeVoice
    text_architecture: TextArchitecture
    mood_mode: MoodMode
    density: DensityLevel
    brand_visibility: BrandVisibility

    reference_image_paths: List[str] = Field(
        default_factory=list,
        description="Uploaded reference image paths available to the run."
    )
    template_id: Optional[str] = Field(None, description="Resolved structured template id, if any.")
    template_image_path: Optional[str] = Field(None, description="Uploaded template image path, if any.")
    logo_image_path: Optional[str] = Field(None, description="Uploaded logo image path, if any.")

    reference_usage_plan: str = Field(
        ...,
        description="How the main reference visual should be used: hero choice, crop, angle, realism treatment, and what must stay true."
    )
    template_usage_plan: Optional[str] = Field(
        None,
        description="How the template should influence style, composition, or hierarchy without changing project identity."
    )
    logo_usage_plan: Optional[str] = Field(
        None,
        description="How the logo should be reproduced and placed."
    )

    asset_decision: AssetDecision = Field(...)
    required_data: List[str] = Field(
        default_factory=list,
        description="Required payload elements such as brand, project, headline, location, RERA, contact."
    )
    conflict_notes: List[str] = Field(
        default_factory=list,
        description="Any brief conflicts and how they were resolved."
    )
    logo_asset_filename: Optional[str] = Field(
        None,
        description="Project-library logo filename if available."
    )
    negative_rules: List[str] = Field(
        default_factory=list,
        description="Critical negative constraints to preserve truth."
    )

## 7. Helpers

In [169]:
def _load_projects() -> Dict[str, Any]:
    return json.loads((DATA_DIR / "projects.json").read_text(encoding="utf-8"))


def _load_brand() -> Dict[str, Any]:
    return json.loads((DATA_DIR / "brand_guidelines.json").read_text(encoding="utf-8"))


def _project_without_images(project: Dict[str, Any]) -> Dict[str, Any]:
    project_copy = dict(project)
    project_copy.pop("image_library", None)
    return project_copy


def _available_asset(project: Dict[str, Any], key: str) -> Optional[Dict[str, Any]]:
    item = project.get("image_library", {}).get(key)
    if item and item.get("available"):
        return item
    return None

## 8. Tools

In [170]:
"""## 8. Tools"""

import json
from pathlib import Path
from typing import Any, Dict, List, Optional


TEMPLATE_LIBRARY: Dict[str, Dict[str, Any]] = {
    "quiet_editorial_launch": {
        "template_id": "quiet_editorial_launch",
        "use_for": ["project_launch", "amenity_spotlight"],
        "poster_archetype": "white_space_editorial_statement",
        "lever_hints": {
            "hero_presentation": "cutout_object",
            "layout_geometry": "open_editorial_field",
            "graphic_layer": ["thin_frame"],
            "type_voice": "premium_serif",
            "text_architecture": "one_statement",
            "mood_mode": "ivory_studio_neutral",
            "density": "ultra_lean",
            "brand_visibility": "whisper",
        },
        "rules": [
            "Use negative space deliberately.",
            "Keep copy very short.",
            "Do not let decorative devices overpower the hero.",
        ],
    },
    "documentary_overlay_update": {
        "template_id": "documentary_overlay_update",
        "use_for": ["construction_update", "site_visit_invite"],
        "poster_archetype": "documentary_presence",
        "lever_hints": {
            "hero_presentation": "candid_presence",
            "layout_geometry": "documentary_crop_overlay",
            "graphic_layer": ["thin_frame", "translucent_panel"],
            "type_voice": "builder_readable",
            "text_architecture": "proposition_first",
            "mood_mode": "crisp_daylight",
            "density": "medium",
            "brand_visibility": "developer_explicit",
        },
        "rules": [
            "Keep the image truthful.",
            "Use overlay treatment for hierarchy, not for decoration.",
            "Do not drift into luxury fantasy styling.",
        ],
    },
    "organic_shape_launch": {
        "template_id": "organic_shape_launch",
        "use_for": ["project_launch", "amenity_spotlight"],
        "poster_archetype": "organic_shape_launch",
        "lever_hints": {
            "hero_presentation": "cutout_object",
            "layout_geometry": "split_panel",
            "graphic_layer": ["organic_shape"],
            "type_voice": "serif_sans_mix",
            "text_architecture": "slogan_first",
            "mood_mode": "warm_muted_premium",
            "density": "medium",
            "brand_visibility": "visible_brand_led",
        },
        "rules": [
            "The shape supports the poster; it should not become the hero.",
            "Stay friendly and premium, not playful.",
            "Keep identity truth stronger than style novelty.",
        ],
    },
}


def _load_projects() -> Dict[str, Any]:
    return json.loads((DATA_DIR / "projects.json").read_text(encoding="utf-8"))


def _load_brand() -> Dict[str, Any]:
    return json.loads((DATA_DIR / "brand_guidelines.json").read_text(encoding="utf-8"))


def _project_without_images(project: Dict[str, Any]) -> Dict[str, Any]:
    project_copy = dict(project)
    project_copy.pop("image_library", None)
    return project_copy


def get_brand_guidelines() -> str:
    """Return the brand guidelines JSON."""
    return json.dumps(_load_brand(), indent=2)


def get_project_details(project_slug: str) -> str:
    """Return project details without the image library."""
    data = _load_projects()
    if project_slug not in data:
        return json.dumps(
            {"error": f"Project '{project_slug}' not found", "available": list(data.keys())},
            indent=2,
        )
    return json.dumps(_project_without_images(data[project_slug]), indent=2)


def get_template_details(template_id: str) -> str:
    """Return structured template guidance."""
    if template_id not in TEMPLATE_LIBRARY:
        return json.dumps(
            {
                "error": f"Template '{template_id}' not found",
                "available_templates": list(TEMPLATE_LIBRARY.keys()),
            },
            indent=2,
        )
    return json.dumps(TEMPLATE_LIBRARY[template_id], indent=2)


def list_asset_candidates(
    project_slug: str,
    post_type: str,
    specific_amenity: Optional[str] = None,
    occasion: Optional[str] = None,
    no_building_image: bool = False,
    brief_text: str = "",
) -> str:
    """
    Return all available project-library assets with enough information for the analyst
    to make the final asset_decision directly and copy values verbatim when needed.
    """
    data = _load_projects()
    if project_slug not in data:
        return json.dumps(
            {"error": f"Project '{project_slug}' not found", "available": list(data.keys())},
            indent=2,
        )

    project = data[project_slug]
    library = project.get("image_library", {})
    logo_asset = project.get("brand_assets", {}).get("logo_primary", {})
    logo_filename = logo_asset.get("filename") if logo_asset.get("available") else None

    available_assets: List[Dict[str, Any]] = []
    unavailable_categories: List[str] = []

    for category, item in library.items():
        if item.get("available"):
            filename = item["filename"]
            available_assets.append(
                {
                    "category": category,
                    "filename": filename,
                    "reference_tag": "{" + filename + "}",
                    "description": item["description"],
                }
            )
        else:
            unavailable_categories.append(category)

    forbidden_categories: List[str] = []
    hard_constraints: List[str] = []

    if no_building_image:
        forbidden_categories = [
            "main_building_exterior",
            "aerial_view",
            "township_overview",
            "entrance_gate",
        ]
        hard_constraints.append(
            "Brief explicitly avoids building imagery. Exclude all building-led categories."
        )
        available_assets = [
            a for a in available_assets if a["category"] not in forbidden_categories
        ]

    if post_type == PostType.amenity_spotlight.value and specific_amenity:
        hard_constraints.append(
            f"Amenity spotlight MUST prioritize the exact amenity or the closest truthful asset for '{specific_amenity}'."
        )

    semantic_guidance = {
        PostType.project_launch.value: (
            "Pick the asset that best anchors a finished launch poster. "
            "Prefer a recognisable hero over a broad aerial unless township-scale is the message."
        ),
        PostType.construction_update.value: (
            "Only a truthful construction_progress asset is valid unless the caller supplies an uploaded reference image."
        ),
        PostType.festival_post.value: (
            "Symbolic or lifestyle assets may outperform building hero shots."
        ),
        PostType.amenity_spotlight.value: (
            "Pick the exact amenity asset if it exists; otherwise pick the closest truthful lifestyle analogue."
        ),
        PostType.site_visit_invite.value: (
            "Prefer assets that imply real on-ground presence, such as sales_gallery or entrance_gate."
        ),
    }.get(post_type, "Pick the asset that best serves the communication job.")

    allowed_categories = [a["category"] for a in available_assets]
    allowed_filenames = [a["filename"] for a in available_assets]
    allowed_reference_tags = [a["reference_tag"] for a in available_assets]

    payload = {
        "project_slug": project_slug,
        "project_name": project["project_name"],
        "project_type": project["type"],
        "post_type": post_type,
        "specific_amenity": specific_amenity,
        "occasion": occasion,
        "no_building_image": no_building_image,
        "brief_text": brief_text,
        "logo_asset_filename": logo_filename,
        "hard_constraints": hard_constraints,
        "forbidden_categories": forbidden_categories,
        "unavailable_categories": unavailable_categories,
        "semantic_guidance": semantic_guidance,
        "available_assets": available_assets,
        "allowed_categories": allowed_categories,
        "allowed_filenames": allowed_filenames,
        "allowed_reference_tags": allowed_reference_tags,
    }

    return json.dumps(payload, indent=2)

"""## 8A. Preload skill materials (no runtime skill tool calls)"""

from pathlib import Path


def _read_text_if_exists(path: Path) -> str:
    return path.read_text(encoding="utf-8") if path.exists() else ""


def _resolve_skill_text(
    skill_name: str,
    fallback_global_name: Optional[str] = None,
) -> str:
    if fallback_global_name and fallback_global_name in globals():
        value = globals()[fallback_global_name]
        if isinstance(value, str) and value.strip():
            return value.strip()

    candidates = [
        SKILLS_DIR / skill_name / "SKILL.md",
        SKILLS_DIR / skill_name / "skill.md",
        SKILLS_DIR / f"{skill_name}.md",
    ]

    for p in candidates:
        txt = _read_text_if_exists(p)
        if txt.strip():
            return txt.strip()

    return ""


def _resolve_skill_reference(skill_name: str, filename: str) -> str:
    candidates = [
        SKILLS_DIR / skill_name / "references" / filename,
        SKILLS_DIR / skill_name / filename,
    ]

    for p in candidates:
        txt = _read_text_if_exists(p)
        if txt.strip():
            return txt.strip()

    return ""


BRIEFLY_SOCIAL_CORE_TEXT = _resolve_skill_text(
    "briefly-social-core",
    fallback_global_name="briefly_social_core_md",
)

BRIEFLY_SOCIAL_ARCHETYPES_TEXT = _resolve_skill_text(
    "briefly-social-archetypes",
    fallback_global_name="briefly_social_archetypes_md",
)

BRIEFLY_SOCIAL_QA_TEXT = _resolve_skill_text(
    "briefly-social-qa",
    fallback_global_name="briefly_social_qa_md",
)

STYLE_PRIMITIVE_MATRIX_TEXT = _resolve_skill_reference(
    "briefly-social-core",
    "style-primitive-matrix.md",
)

STYLE_FAMILY_MAP_TEXT = _resolve_skill_reference(
    "briefly-social-archetypes",
    "style-family-map.md",
)

In [171]:
"""## 8A. Preload skill materials (no runtime skill tool calls)"""

from pathlib import Path


def _read_text_if_exists(path: Path) -> str:
    return path.read_text(encoding="utf-8") if path.exists() else ""


def _resolve_skill_text(
    skill_name: str,
    fallback_global_name: Optional[str] = None,
) -> str:
    if fallback_global_name and fallback_global_name in globals():
        value = globals()[fallback_global_name]
        if isinstance(value, str) and value.strip():
            return value.strip()

    candidates = [
        SKILLS_DIR / skill_name / "SKILL.md",
        SKILLS_DIR / skill_name / "skill.md",
        SKILLS_DIR / f"{skill_name}.md",
    ]

    for p in candidates:
        txt = _read_text_if_exists(p)
        if txt.strip():
            return txt.strip()

    return ""


def _resolve_skill_reference(skill_name: str, filename: str) -> str:
    candidates = [
        SKILLS_DIR / skill_name / "references" / filename,
        SKILLS_DIR / skill_name / filename,
    ]

    for p in candidates:
        txt = _read_text_if_exists(p)
        if txt.strip():
            return txt.strip()

    return ""


BRIEFLY_SOCIAL_CORE_TEXT = _resolve_skill_text(
    "briefly-social-core",
    fallback_global_name="briefly_social_core_md",
)

BRIEFLY_SOCIAL_ARCHETYPES_TEXT = _resolve_skill_text(
    "briefly-social-archetypes",
    fallback_global_name="briefly_social_archetypes_md",
)

BRIEFLY_SOCIAL_QA_TEXT = _resolve_skill_text(
    "briefly-social-qa",
    fallback_global_name="briefly_social_qa_md",
)

STYLE_PRIMITIVE_MATRIX_TEXT = _resolve_skill_reference(
    "briefly-social-core",
    "style-primitive-matrix.md",
)

STYLE_FAMILY_MAP_TEXT = _resolve_skill_reference(
    "briefly-social-archetypes",
    "style-family-map.md",
)

## 9. Agents

In [175]:
"""## 9. Agents"""

from __future__ import annotations

from agno.skills import Skills, LocalSkills

MODEL_ID = "google/gemini-2.5-flash-lite"


class VerifierInput(BaseModel):
    brief_analysis: BriefAnalysis
    crafted_prompt: CraftedPrompt


def build_model() -> OpenRouterResponses:
    return OpenRouterResponses(
        id=MODEL_ID,
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )


def build_skills() -> Skills:
    return Skills(loaders=[LocalSkills(str(SKILLS_DIR))])


analyst = Agent(
    name="Brief Analyst",
    model=build_model(),
    input_schema=AnalystInput,
    tools=[
        get_project_details,
        get_brand_guidelines,
        get_template_details,
        list_asset_candidates,
    ],
    skills=build_skills(),
    output_schema=BriefAnalysis,
    debug_mode=True,
    markdown=True,
    instructions=[
        "You are a structured brief analyst for a real-estate social-media prompt pipeline.",
        "Return ONLY structured data matching the schema.",
        "",
        "You have access to Briefly Social skills.",
        "Call get_skill_instructions('briefly-social-core') before deciding delivery_mode.",
        "Call get_skill_instructions('briefly-social-archetypes') before choosing poster_archetype and lever bundle.",
        "Do NOT call get_skill_script. There are no scripts in these skills.",
        "",
        "Workflow:",
        "1. Read the supplied project_slug. It is canonical for this run.",
        "2. If selected_post_type is provided, use it. Otherwise infer post_type from the brief.",
        "3. Call get_project_details(project_slug) and get_brand_guidelines().",
        "4. If template_id is provided, call get_template_details(template_id).",
        "5. The attached images correspond to reference_image_paths, template_image_path, and logo_image_path in the input.",
        "6. If reference_image_paths are provided, use vision to understand what the image(s) show, decide the strongest hero, and decide crop / angle / realism treatment.",
        "7. Treat template_image_path as a style and composition cue only. It must not change project identity.",
        "8. Treat logo_image_path as an exact brand mark. Preserve it faithfully.",
        "9. If reference_image_paths are provided, prefer uploaded_reference as the primary asset source unless the brief clearly needs a project-library asset instead.",
        "10. If reference_image_paths are empty and the post type requires a truthful visual anchor, you must use a project-library asset when one exists.",
        "11. For construction_update, if no uploaded reference image is provided, call list_asset_candidates(project_slug, post_type, specific_amenity, occasion, no_building_image, brief_text=user_brief) and choose the truthful construction_progress asset from the project library when available.",
        "12. Do not use asset_decision.source='none' for construction_update when a project-library construction asset exists.",
        "13. If you need a project-library asset, call list_asset_candidates(project_slug, post_type, specific_amenity, occasion, no_building_image, brief_text=user_brief).",
        "14. If asset_decision.source='project_library', copy category, filename, and reference_tag verbatim from the tool output.",
        "15. If asset_decision.source='uploaded_reference', copy filepath verbatim from the input, set filename to the basename of the chosen path, and set reference_tag=null.",
        "16. Use asset_decision.source='none' only when visual_mode is graphic_led or no truthful visual anchor fits.",
        "",
        "Non-negotiable rules:",
        "- Do not fabricate filenames, filepaths, reference_tags, or template ids.",
        "- Do not let the template image override reference-image truth.",
        "- Do not let the logo become stylized, distorted, or rewritten.",
        "- Do not treat post_type as design style.",
        "",
        "Defaults:",
        "- Default delivery_mode to finished_poster unless the brief explicitly asks for a base visual.",
        "- For finished_poster mode, default text_policy to exact_text.",
        "- social_post=1:1, story=9:16, banner=16:9 unless the brief explicitly says otherwise.",
        "- visual_mode: asset_faithful for documentary truth, editorialized_truth for most premium posters, graphic_led only when no truthful asset fits.",
        "",
        "You must fill:",
        "- reference_image_paths",
        "- template_id / template_image_path / logo_image_path if present",
        "- reference_usage_plan",
        "- template_usage_plan when a template exists",
        "- logo_usage_plan when a logo image exists",
        "- conflict_notes when brief, template, and truth pull in different directions",
    ],
)

crafter = Agent(
    name="Prompt Crafter",
    model=build_model(),
    input_schema=BriefAnalysis,
    output_schema=CraftedPrompt,
    skills=build_skills(),
    debug_mode=True,
    markdown=True,
    instructions=[
        "You are a real-estate image prompt engineer.",
        "Return ONLY structured data matching the schema.",
        "",
        "You have access to Briefly Social skills.",
        "Call get_skill_instructions('briefly-social-core') when needed.",
        "Call get_skill_instructions('briefly-social-archetypes') when needed.",
        "Call get_skill_reference('briefly-social-core', 'style-primitive-matrix.md') only when needed.",
        "Call get_skill_reference('briefly-social-archetypes', 'style-family-map.md') only when needed.",
        "Do NOT call get_skill_script. There are no scripts in these skills.",
        "",
        "Rules:",
        "- The attached images correspond to reference_image_paths, template_image_path, and logo_image_path in the analysis.",
        "- If asset_decision.reference_tag exists, start the prompt with it.",
        "- If asset_decision.source='uploaded_reference', do not invent a filename; rely on the attached reference image and the reference_usage_plan.",
        "- If asset_decision.source='project_library', the prompt must stay anchored to the selected category and reference_tag.",
        "- Preserve actual architecture, amenity design, construction state, and recognisable identity.",
        "- You may improve crop, camera angle, clarity, realism, and editorial finish when the reference_usage_plan allows it.",
        "- template_usage_plan shapes style and composition only. It must not replace project truth.",
        "- logo_usage_plan means the logo image should be reproduced exactly and placed with discipline.",
        "- Keep prompts dense, visual, and compositional.",
        "- If delivery_mode=finished_poster, write a finished poster prompt, not a scenic base-image prompt.",
        "- poster_archetype and levers must be visible in the final prompt.",
        "- If text_policy=exact_text, include the exact text and integrate it into the poster composition.",
        "- Do not write a generic prompt for construction_update without a truthful primary asset.",
    ],
)

verifier = Agent(
    name="Prompt Verifier",
    model=build_model(),
    input_schema=VerifierInput,
    output_schema=VerificationResult,
    debug_mode=True,
    markdown=True,
    instructions=[
        "You are the final QA gate for real-estate image prompts.",
        "Return ONLY structured data matching the schema.",
        "",
        "Check:",
        "- truthfulness and recognisability",
        "- correct use of the reference image",
        "- template staying style-only instead of identity-changing",
        "- exact-logo requirement when a logo image is present",
        "- business-job fit, hierarchy, style-family fidelity, and anti-repetition quality",
        "",
        "Revision rules:",
        "- Reject outputs where construction_update has no truthful primary asset while a project-library construction asset exists.",
        "- Reject outputs that describe a generic facade crop or generic documentary poster without a grounded primary asset.",
        "- Reject outputs where the template image is likely to override project identity.",
        "- Reject outputs where the logo could be distorted or rewritten.",
        "- Reject outputs that feel like a plain hero image plus normal text.",
        "- Reject outputs where the selected style family is named but not visibly executed.",
        "",
        "Approval rules:",
        "- Approve only if the prompt is grounded, specific, recognisable, and poster-grade.",
        "- If the prompt is already good, keep it with minimal edits.",
        "- If the prompt is weak, revise the prompt and negative constraints directly instead of describing the problem only.",
    ],
)

DEBUG Loaded 3 skills from /content/skills

DEBUG Loaded 3 total skills

DEBUG Loaded 3 skills from /content/skills

DEBUG Loaded 3 total skills

## 10. Orchestration

In [176]:
"""## 10. Orchestration"""

import json
from pathlib import Path
from agno.media import Image


def _normalize_post_type(post_type: Optional[object]) -> Optional[PostType]:
    if post_type is None:
        return None
    if isinstance(post_type, PostType):
        return post_type
    try:
        return PostType(str(post_type))
    except ValueError as e:
        valid = [p.value for p in PostType]
        raise ValueError(f"Invalid post_type '{post_type}'. Valid values are: {valid}") from e


def _unique_existing_paths(paths: List[str]) -> List[str]:
    cleaned: List[str] = []
    for raw in paths:
        if not raw:
            continue
        p = str(Path(raw))
        if not Path(p).exists():
            raise FileNotFoundError(f"Image not found: {p}")
        if p not in cleaned:
            cleaned.append(p)
    return cleaned


def _build_run_images(analyst_input: AnalystInput) -> List[Image]:
    ordered_paths: List[str] = []
    ordered_paths.extend(analyst_input.reference_image_paths)
    if analyst_input.template_image_path:
        ordered_paths.append(analyst_input.template_image_path)
    if analyst_input.logo_image_path:
        ordered_paths.append(analyst_input.logo_image_path)

    return [Image(filepath=p) for p in _unique_existing_paths(ordered_paths)]


def _step_session_id(base_session_id: Optional[str], suffix: str) -> Optional[str]:
    if not base_session_id:
        return None
    return f"{base_session_id}:{suffix}"


def _repair_analysis_grounding(analysis: BriefAnalysis) -> BriefAnalysis:
    """
    Hard safety repair:
    If the analyst returns source='none' for construction_update with no uploaded reference image,
    but the project library has a truthful construction_progress asset, force the analysis back onto
    that grounded asset.
    """
    if analysis.post_type != PostType.construction_update:
        return analysis

    if analysis.reference_image_paths:
        return analysis

    if analysis.asset_decision.source != "none":
        return analysis

    payload = json.loads(
        list_asset_candidates(
            project_slug=analysis.project_slug,
            post_type=analysis.post_type.value if hasattr(analysis.post_type, "value") else str(analysis.post_type),
            specific_amenity=analysis.specific_amenity,
            occasion=analysis.occasion,
            no_building_image=analysis.no_building_image,
            brief_text=analysis.business_job,
        )
    )

    available_assets = payload.get("available_assets", [])
    construction_asset = None
    for asset in available_assets:
        if asset.get("category") == "construction_progress":
            construction_asset = asset
            break

    if not construction_asset:
        return analysis

    analysis.asset_decision.source = "project_library"
    analysis.asset_decision.category = construction_asset["category"]
    analysis.asset_decision.filename = construction_asset["filename"]
    analysis.asset_decision.filepath = None
    analysis.asset_decision.reference_tag = construction_asset["reference_tag"]
    analysis.asset_decision.reason = (
        "Repaired grounding: construction_update requires a truthful construction asset, "
        "and the project library contains construction_progress."
    )
    analysis.asset_decision.fallback_categories = ["main_building_exterior"]
    analysis.asset_decision.recommended_poster_archetype = PosterArchetype.documentary_presence.value

    analysis.poster_archetype = PosterArchetype.documentary_presence
    analysis.hero_presentation = HeroPresentation.candid_presence
    analysis.layout_geometry = LayoutGeometry.documentary_crop_overlay
    analysis.mood_mode = MoodMode.crisp_daylight
    analysis.visual_mode = VisualMode.asset_faithful
    analysis.delivery_mode = DeliveryMode.finished_poster
    analysis.text_policy = TextPolicy.exact_text

    analysis.reference_usage_plan = (
        f"Use the project-library construction asset {construction_asset['filename']} as the truthful primary anchor. "
        "Show real progress, keep the construction state accurate, and use crop / hierarchy / overlay treatment to make it premium without falsifying the site."
    )

    repair_note = (
        "System repair applied: analyst returned source='none' for construction_update even though a project-library "
        "construction_progress asset exists. Grounding was corrected automatically."
    )
    if repair_note not in analysis.conflict_notes:
        analysis.conflict_notes.append(repair_note)

    return analysis


def run_pipeline(
    project_slug: str,
    user_brief: str,
    post_type: Optional[object] = None,
    reference_image_paths: Optional[List[str]] = None,
    reference_image_note: Optional[str] = None,
    template_id: Optional[str] = None,
    template_image_path: Optional[str] = None,
    template_note: Optional[str] = None,
    logo_image_path: Optional[str] = None,
    logo_note: Optional[str] = None,
    session_id: Optional[str] = None,
    print_steps: bool = True,
) -> Dict[str, Any]:
    analyst_input = AnalystInput(
        project_slug=project_slug,
        user_brief=user_brief,
        selected_post_type=_normalize_post_type(post_type),
        reference_image_paths=reference_image_paths or [],
        reference_image_note=reference_image_note,
        template_id=template_id,
        template_image_path=template_image_path,
        template_note=template_note,
        logo_image_path=logo_image_path,
        logo_note=logo_note,
    )

    run_images = _build_run_images(analyst_input)

    analyst_kwargs: Dict[str, Any] = {
        "input": analyst_input,
        "session_id": _step_session_id(session_id, "analyst"),
    }
    if run_images:
        analyst_kwargs["images"] = run_images
    analyst_response = analyst.run(**analyst_kwargs)
    analysis = analyst_response.content

    # Hard grounding repair for construction updates
    analysis = _repair_analysis_grounding(analysis)

    crafter_kwargs: Dict[str, Any] = {
        "input": analysis,
        "session_id": _step_session_id(session_id, "crafter"),
    }
    if run_images:
        crafter_kwargs["images"] = run_images
    crafter_response = crafter.run(**crafter_kwargs)
    crafted = crafter_response.content

    verifier_input = VerifierInput(
        brief_analysis=analysis,
        crafted_prompt=crafted,
    )
    verifier_kwargs: Dict[str, Any] = {
        "input": verifier_input,
        "session_id": _step_session_id(session_id, "verifier"),
    }
    if run_images:
        verifier_kwargs["images"] = run_images
    verifier_response = verifier.run(**verifier_kwargs)
    verified = verifier_response.content

    if print_steps:
        print("=" * 90); print("STEP 1 — ANALYSIS"); print("=" * 90)
        print(analysis.model_dump_json(indent=2) if hasattr(analysis, "model_dump_json") else analysis)
        print("\n" + "=" * 90); print("STEP 2 — CRAFTING"); print("=" * 90)
        print(crafted.model_dump_json(indent=2) if hasattr(crafted, "model_dump_json") else crafted)
        print("\n" + "=" * 90); print("STEP 3 — VERIFICATION"); print("=" * 90)
        print(verified.model_dump_json(indent=2) if hasattr(verified, "model_dump_json") else verified)

    return {
        "session_id": session_id or verifier_response.session_id,
        "analysis": analysis.model_dump() if hasattr(analysis, "model_dump") else analysis,
        "crafted": crafted.model_dump() if hasattr(crafted, "model_dump") else crafted,
        "verified": verified.model_dump() if hasattr(verified, "model_dump") else verified,
    }


def generate_image_prompt(
    project_slug: str,
    user_brief: str,
    post_type: Optional[object] = None,
    reference_image_paths: Optional[List[str]] = None,
    reference_image_note: Optional[str] = None,
    template_id: Optional[str] = None,
    template_image_path: Optional[str] = None,
    template_note: Optional[str] = None,
    logo_image_path: Optional[str] = None,
    logo_note: Optional[str] = None,
    session_id: Optional[str] = None,
    print_steps: bool = True,
) -> Dict[str, Any]:
    result = run_pipeline(
        project_slug=project_slug,
        user_brief=user_brief,
        post_type=post_type,
        reference_image_paths=reference_image_paths,
        reference_image_note=reference_image_note,
        template_id=template_id,
        template_image_path=template_image_path,
        template_note=template_note,
        logo_image_path=logo_image_path,
        logo_note=logo_note,
        session_id=session_id,
        print_steps=print_steps,
    )

    analysis = result["analysis"]
    crafted = result["crafted"]
    verified = result["verified"]

    print("\n" + "=" * 90); print("FINAL APPROVED OUTPUT"); print("=" * 90)
    print(f"Session: {result['session_id']}")
    print(f"Project: {analysis['project_name']} ({analysis['project_slug']})")
    print(f"Post Type: {analysis['post_type']} | Format: {analysis['format_type']} ({analysis['aspect_ratio']})")
    print(f"Delivery: {analysis['delivery_mode']} | Archetype: {analysis['poster_archetype']}")
    print(f"Approved: {verified['approved']}")
    print("\nPrompt:\n" + verified["revised_prompt"])
    print("\nNegative:\n" + verified["revised_negative"])

    return result

## 11. Example runs

In [179]:

"""## 11. Example runs"""

if __name__ == "__main__":
    generate_image_prompt(
        project_slug="horizon-heights",
        post_type="project_launch",
        user_brief="Create a premium 1:1 project launch poster for Horizon Heights. Make it aspirational, recognisable, and poster-grade.",
        reference_image_paths=[REFERENCE_IMAGE_PATH] if REFERENCE_IMAGE_PATH else None,
        reference_image_note=REFERENCE_IMAGE_NOTE if REFERENCE_IMAGE_PATH else None,
        template_id=TEMPLATE_ID,
        template_image_path=TEMPLATE_IMAGE_PATH,
        template_note=TEMPLATE_NOTE if TEMPLATE_IMAGE_PATH or TEMPLATE_ID else None,
        logo_image_path=LOGO_IMAGE_PATH,
        logo_note=LOGO_NOTE if LOGO_IMAGE_PATH else None,
        print_steps=True,
    )

DEBUG ***** Session ID: 96ebd3ba-b146-45c7-8561-b6d0fc53c7db *****

DEBUG ***************** Agent ID: brief-analyst ******************

DEBUG Creating new AgentSession: 96ebd3ba-b146-45c7-8561-b6d0fc53c7db

DEBUG Setting Model.response_format to Agent.output_schema

DEBUG ** Agent Run Start: 0dc99ee7-a7d6-4aca-b629-ede78062a1e2 ***

...

DEBUG ************************  METRICS  *************************

DEBUG ------------ OpenRouter Responses Response End -------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: dc6e2242-fb2a-4f3c-86f7-d1501b4b4572 ****

STEP 1 — ANALYSIS
{
  "project_slug": "horizon-heights",
  "project_name": "Horizon Heights",
  "post_type": "project_launch",
  "business_job": "Launch Horizon Heights with an aspirational and recognizable poster.",
  "persuasion_modes": [
    "aspirational"
  ],
  "style_modifiers": [
    "recognisable"
  ],
  "format_type": "social_post",
  "aspect_ratio": "1:1",
  "occasion": null,
  "specific_amenity": null,
  "no_building_image": false,
  "visual_mode": "editorialized_truth",
  "delivery_mode": "finished_poster",
  "text_policy": "exact_text",
  "poster_archetype": "organic_shape_launch",
  "hero_presentation": "architecture_with_environment",
  "layout_geometry": "lower_hero_upper_copy",
  "graphic_layer": [
    "organic_shape"
  ],
  "type_voice": "modern_sans",
  "text_architecture": "slogan_first",
  "mood_mode": "crisp_daylight",
  "density": "medium",
  "brand_visibility": "whisper",
  "reference_image_paths": [],
  "template_id": null,
  "template_image_path": null,
  "log